# Assignment 4: Build an MLOps Pipeline for Your GNN

## 1. Overview and Architecture
This assignment establishes a production-grade, reproducible, tracked, and deployable GNN pipeline.

### Key Components:
1. **Untracked Baseline**: Demonstrates the failure modes of unversioned scripts.
2. **Experiment Tracking**: MLflow tracking across 3 distinct runs (parameters, seeds, metrics, artifacts).
3. **Model Packaging & Reload Consistency**: Serialization of state dict, feature schemas, and bitwise reload verification.
4. **FastAPI Service & Failure Handling**: Serving existing nodes, unseen nodes, and catching bad schemas.
5. **Model Registry & Promotion Gate**: Enforcing a strict validation improvement threshold and blocking regressions.
6. **Drift Detection**: Monitoring graph degree shift and embedding cosine distance.
7. **Automated Rollback & CI/CD**: Demonstrating production restoration and GitHub Actions workflows.

In [1]:
import os
import sys
sys.path.insert(0, r"D:\gnn-practical-assignments")

import pytest
from assignment4_mlops.tracking.train_tracked import run_all_experiments
from assignment4_mlops.tracking.registry_gate import demonstrate_registry_gate
from assignment4_mlops.monitoring.drift_detector import run_drift_demonstration
from assignment4_mlops.tracking.rollback_demo import rollback_production
print("Assignment 4 environment loaded.")

C:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026/10/10 21:23:35 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Assignment 4 environment loaded.


## 2. MLflow Experiment Tracking & Packaging
We execute 3 tracked experiments, record hyperparameter metrics to SQLite MLflow, and package the artifacts.

In [2]:
run_all_experiments()

RUNNING 3 EXPERIMENTS WITH MLFLOW TRACKING


[OK] Reload reproducibility verified: max diff = 0.00000000e+00
[TRACKED] Run 'model_v1_baseline' saved to MLflow (Val F1: 0.7982, Test Acc: 82.70%)


[OK] Reload reproducibility verified: max diff = 0.00000000e+00
[TRACKED] Run 'model_v2_high_capacity' saved to MLflow (Val F1: 0.7937, Test Acc: 81.50%)


[OK] Reload reproducibility verified: max diff = 0.00000000e+00
[TRACKED] Run 'model_v3_high_reg' saved to MLflow (Val F1: 0.7974, Test Acc: 81.60%)
[SUCCESS] All 3 experiments tracked in MLflow and packaged in artifacts directory.


## 3. Model Registry & Validation Gate
We test the promotion gate: automatic promotion of the baseline and blocking of regressive candidate models.

In [3]:
demonstrate_registry_gate()

DEMONSTRATING MODEL REGISTRY & VALIDATION GATE
[REGISTRY] Model 'model_v1_baseline' promoted and deployed to active production directory.

Case 1 (First Model):
  Candidate: model_v1_baseline -> Decision: PROMOTED
  Reason:    First model in registry - automatically promoted as production baseline.

Case 2 (Blocked Regression):
  Candidate: model_v2_high_capacity -> Decision: BLOCKED_REGRESSION
  Reason:    Candidate regressed or tied production (0.7937 vs 0.7982). Promotion blocked.

[SUCCESS] Promotion gate successfully protected production from regressive model.


## 4. API Endpoints, Edge Cases & Consistency Tests
We run the full pytest suite verifying prediction consistency, isolated nodes, and schema validation.

In [4]:
pytest.main(["-v", "-s", r"D:\gnn-practical-assignments\assignment4_mlops\tests\test_api.py"])

============================= test session starts =============================
platform win32 -- Python 3.14.3, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\hp\AppData\Local\Programs\Python\Python314\python.exe
cachedir: .pytest_cache
rootdir: D:\gnn-practical-assignments\assignment4_mlops
plugins: platformdirs-4.12.4, anyio-4.15.1
collecting ... 

collected 8 items

tests/test_api.py::test_health_endpoint 


[OK] Health check passed.
PASSED


tests/test_api.py::test_offline_vs_api_prediction_consistency 

[OK] Offline and API predictions match perfectly.
PASSED

...


PASSED


tests/test_api.py::test_failure_unknown_target_id 

[OK] Unknown target ID caught with HTTP 400.
PASSED



============================== warnings summary ===============================
C:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\_pytest\config\__init__.py:1345
  C:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\_pytest\config\__init__.py:1345: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

C:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\fastapi\testclient.py:1
  C:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

tests/test_api.py: 22 warnings
  C:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch_geometric\inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_p

<ExitCode.OK: 0>

## 5. Drift Monitoring (Graph Structure & Embeddings)
We compare in-distribution snapshots against out-of-distribution batches and monitor degree shift and cosine distance.

In [5]:
run_drift_demonstration()

MONITORING: GRAPH-STRUCTURE & EMBEDDING DRIFT DETECTION

--- Drift Assessment: Snapshot 'Normal_Production_Batch' ---
Graph Nodes / Edges:       500 / 418
Average Degree:            0.84 (Baseline: 3.90) -> Relative Shift: 78.6%
Structural Drift Alert:    [ALERT: DEGREE SHIFT]
Embedding Cosine Distance: 0.0006 (Threshold: 0.20)
Embedding Drift Alert:     [NORMAL]
Final Decision:            *** DRIFT DETECTED ***

--- Drift Assessment: Snapshot 'Drifted_Out_Of_Distribution_Batch' ---
Graph Nodes / Edges:       500 / 4
Average Degree:            0.01 (Baseline: 3.90) -> Relative Shift: 99.8%
Structural Drift Alert:    [ALERT: DEGREE SHIFT]
Embedding Cosine Distance: 0.0225 (Threshold: 0.20)
Embedding Drift Alert:     [NORMAL]
Final Decision:            *** DRIFT DETECTED ***

EXPLANATION: WHY RAW EMBEDDINGS ACROSS DIFFERENT MODEL VERSIONS CANNOT BE COMPARED:
1. Rotational & Permutation Invariance: Deep neural networks have many equivalent global minima.
   Two training runs produce weigh

## 6. Automated Rollback Simulation
We demonstrate rolling back production to the previous known stable version.

In [6]:
rollback_production()

AUTOMATED PRODUCTION ROLLBACK DEMONSTRATION
[NOTICE] Rollback simulation: Injecting previous known stable checkpoint into registry history.
Current live model:      model_v1_baseline
Rolling back to model:  model_v1_baseline
[SUCCESS] Production successfully restored to 'model_v1_baseline'.


{'decision': 'ROLLED_BACK',
 'candidate': 'model_v1_baseline',
 'reason': 'Manual/Automated rollback triggered from model_v1_baseline to model_v1_baseline',
 'metric': 0.7982}